In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

True
NVIDIA GeForce RTX 2050


In [ ]:
import os

# Use only 1 GPU if available. If CUDA is unavailable, TimesFM will run on CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import TimesFmModelForPrediction # from HuggingFace transformers pip version 4.57.6

TIMESFM_MODEL_ID = "google/timesfm-2.0-500m-pytorch"
TIMESFM_MODEL_NAME = "TimesFM"
TIMESFM_CONTEXT_LENGTH = None  # None uses the model default, as in timesfm.ipynb

# Same loading pattern as timesfm.ipynb.
timesfm_model = TimesFmModelForPrediction.from_pretrained(
    TIMESFM_MODEL_ID,
    attn_implementation="sdpa",
    device_map="auto",
)


In [3]:
from pprint import pprint
from copy import deepcopy
import os

import pandas as pd
import numpy as np
import torch

from utilsforecast.losses import mase
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive

from src.meta.arima._data_reader import ModelIO
from src.chronos_data import ChronosDataset


def timesfm_frequency_id(freq):
    """Map pandas/StatsForecast frequency strings to TimesFM frequency ids."""
    freq = str(freq).upper()
    if freq.endswith("MS"):
        return 1
    if freq.endswith(("H", "T", "MIN", "D", "B", "U", "S")):
        return 0
    if freq.endswith(("W", "M")) or freq.startswith("W-") or (freq.startswith("M") and len(freq) == 2):
        return 1
    if freq.endswith(("Y", "Q", "A")) or freq.startswith(("Y-", "Q-", "A-")):
        return 2
    raise ValueError(f"Invalid TimesFM frequency: {freq}")


def timesfm_predict_df(
    history_df,
    model,
    prediction_length,
    freq,
    model_col=TIMESFM_MODEL_NAME,
    future_ds=None,
):
    """Forecast a long dataframe with columns unique_id, ds, y using TimesFM."""
    required_cols = ["unique_id", "ds", "y"]
    missing_cols = [col for col in required_cols if col not in history_df.columns]
    if missing_cols:
        raise ValueError(f"history_df is missing columns: {missing_cols}")

    timesfm_input = history_df[required_cols].copy()
    timesfm_input["ds"] = pd.to_datetime(timesfm_input["ds"])
    timesfm_input = timesfm_input.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    ids = []
    grouped_inputs = []
    forecast_input_tensor = []
    for uid, uid_df in timesfm_input.groupby("unique_id", sort=False):
        ids.append(uid)
        grouped_inputs.append(uid_df)
        forecast_input_tensor.append(
            torch.tensor(uid_df["y"].to_numpy(dtype=np.float32), dtype=torch.float32).to(model.device)
        )

    frequency_input_tensor = torch.tensor(
        [timesfm_frequency_id(freq)] * len(forecast_input_tensor),
        dtype=torch.long,
    ).to(model.device)

    with torch.no_grad():
        if TIMESFM_CONTEXT_LENGTH is None:
            outputs = model(forecast_input_tensor, frequency_input_tensor)
        else:
            outputs = model(
                forecast_input_tensor,
                frequency_input_tensor,
                forecast_context_len=TIMESFM_CONTEXT_LENGTH,
            )
        point_forecast = outputs.mean_predictions.float().cpu().numpy()

    frames = []
    future_ds = None if future_ds is None else pd.to_datetime(pd.Series(future_ds))

    for i, (uid, uid_history) in enumerate(zip(ids, grouped_inputs)):
        y_pred = np.asarray(point_forecast[i, :prediction_length]).reshape(-1)

        if future_ds is not None and len(ids) == 1:
            ds_values = future_ds.iloc[: len(y_pred)].to_numpy()
        else:
            last_ds = uid_history["ds"].max()
            ds_values = pd.date_range(
                start=last_ds,
                periods=prediction_length + 1,
                freq=freq,
            )[1 : len(y_pred) + 1]

        frames.append(
            pd.DataFrame(
                {
                    "unique_id": uid,
                    "ds": ds_values,
                    model_col: y_pred[: len(ds_values)],
                }
            )
        )

    return pd.concat(frames, ignore_index=True)


OVERRIDE_DS = False
algorithm = "catboost"
source = "m4_monthly"
FILENAME = f"assets/trained_metaarima_{source}_{algorithm}.joblib.gz"
meta_arima = ModelIO.load_model(FILENAME)

target = "monash_tourism_monthly"

df, horizon, _, freq, seas_len = ChronosDataset.load_everything(target)
train, test = ChronosDataset.time_wise_split(df, horizon)

sf_models = [AutoARIMA(season_length=seas_len), SeasonalNaive(season_length=seas_len)]
model_names = ["MetaARIMA", "AutoARIMA", "SeasonalNaive", TIMESFM_MODEL_NAME]

uids = train["unique_id"].unique().tolist()

results, predictions = [], []
for uid in uids:
    print(uid)

    df_uid_tr = train.query(f'unique_id=="{uid}"').reset_index(drop=True)
    df_uid_ts = test.query(f'unique_id=="{uid}"').reset_index(drop=True)
    if df_uid_ts.isna().any()["y"]:
        continue

    meta_arima.fit(df_uid_tr, freq=freq, seas_length=seas_len)

    fcst_ma = meta_arima.predict(h=horizon)

    sf = StatsForecast(models=deepcopy(sf_models), freq=freq)
    sf.fit(df_uid_tr)

    fcst_aa = sf.forecast(h=horizon)

    fcst_tsfm1 = timesfm_predict_df(
        df_uid_tr,
        model=timesfm_model,
        prediction_length=horizon,
        freq=freq,
        future_ds=df_uid_ts["ds"],
    )
    fcst_tsfm1 = fcst_tsfm1[["unique_id", "ds", TIMESFM_MODEL_NAME]]

    print(fcst_tsfm1.head())

    if OVERRIDE_DS:
        fcst_ma["ds"] = df_uid_ts["ds"].values
        fcst_aa["ds"] = df_uid_ts["ds"].values
        fcst_tsfm1["ds"] = df_uid_ts["ds"].values

    uid_test = df_uid_ts.merge(fcst_ma, on=["unique_id", "ds"])
    uid_test = uid_test.merge(fcst_aa, on=["unique_id", "ds"])
    uid_test = uid_test.merge(fcst_tsfm1, on=["unique_id", "ds"])

    print(uid_test.head())

    err = mase(
        df=uid_test, models=model_names, seasonality=seas_len, train_df=df_uid_tr
    )

    pprint(err)

    predictions.append(uid_test)
    results.append(err)
    results_df = pd.concat(results)
    print(results_df.mean(numeric_only=True))
    print(results_df.median(numeric_only=True))

results_df = pd.concat(results)
predictions_df = pd.concat(predictions).reset_index(drop=True)
print(results_df.mean(numeric_only=True))
print(results_df.median(numeric_only=True))

output_dir = "assets/results/timesfm"
os.makedirs(output_dir, exist_ok=True)
results_df.to_csv(f"{output_dir}/scores,{target}.csv", index=False)
predictions_df.to_csv(f"{output_dir}/predictions,{target}.csv", index=False)


T000000
  unique_id         ds      TimesFM
0   T000000 1993-08-31  6604.729492
1   T000000 1993-09-30  4250.816406
2   T000000 1993-10-31  3006.237793
3   T000000 1993-11-30  2185.868896
4   T000000 1993-12-31  2456.317871
  unique_id         ds         y    MetaARIMA    AutoARIMA  SeasonalNaive  \
0   T000000 1993-08-31  6857.800  6596.216691  6667.752787      6611.1150   
1   T000000 1993-09-30  4346.090  4306.520053  4227.479462      4150.2395   
2   T000000 1993-10-31  3154.730  2975.831188  2969.658623      2841.0000   
3   T000000 1993-11-30  2142.210  1854.537555  1938.663654      1813.4400   
4   T000000 1993-12-31  2375.725  2441.042711  2371.430481      2261.0800   

       TimesFM  
0  6604.729492  
1  4250.816406  
2  3006.237793  
3  2185.868896  
4  2456.317871  
  unique_id  MetaARIMA  AutoARIMA  SeasonalNaive   TimesFM
0   T000000   1.372374   1.286285       1.649169  0.967977
MetaARIMA        1.372374
AutoARIMA        1.286285
SeasonalNaive    1.649169
TimesFM        